In [ ]:
# =========================
# INSTALL DEPENDENCIES
# =========================
# Run this once: pip install psycopg2-binary sqlalchemy pandas openpyxl

# =========================
# IMPORTS
# =========================
from sqlalchemy import create_engine, text
import pandas as pd
from datetime import datetime
import os

# =========================
# DATABASE CONFIGURATION
# =========================
user = "postgres"
password = "YOUR_PASSWORD_HERE"  # UPDATE THIS!
host = "localhost"
port = "5432"
database = "knee_database1"

# Create engine
engine = create_engine(
    f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}",
    future=True
)

# =========================
# VERIFY CONNECTION
# =========================
print("Testing database connection...")
with engine.connect() as conn:
    db = conn.execute(text("SELECT current_database();")).scalar()
    print(f"✅ Connected to database: {db}")

# =========================
# HELPER FUNCTIONS
# =========================
def clean_measurement(val):
    """Clean and validate measurement values"""
    if pd.isna(val):
        return None
    if isinstance(val, str):
        val = val.strip()
        if val == "" or val.lower() == "n/a":
            return None
    try:
        return float(val)
    except (ValueError, TypeError):
        return None

def parse_date(val):
    """Parse date from various formats"""
    if pd.isna(val):
        return None
    if isinstance(val, datetime):
        return val.date()
    if isinstance(val, str):
        for fmt in ["%Y-%m-%d", "%m/%d/%Y", "%d/%m/%Y"]:
            try:
                return datetime.strptime(val, fmt).date()
            except ValueError:
                continue
    return None

def validate_demographics(row, row_num, warnings):
    """Validate demographic data and collect warnings"""
    valid = True
    
    # Validate Age
    age = row.get("Age")
    if pd.notna(age):
        try:
            age_val = int(age)
            if age_val < 0 or age_val > 120:
                warnings.append(f"Row {row_num}: Invalid age {age_val} (must be 0-120)")
                valid = False
        except (ValueError, TypeError):
            warnings.append(f"Row {row_num}: Age must be a number")
            valid = False
    
    # Validate Gender
    gender = row.get("Gender")
    if pd.notna(gender):
        try:
            gender_val = int(gender)
            if gender_val not in [0, 1, 2]:  # Adjust based on your coding scheme
                warnings.append(f"Row {row_num}: Invalid gender code {gender_val}")
        except (ValueError, TypeError):
            warnings.append(f"Row {row_num}: Gender must be a number")
    
    # Validate Weight
    weight_lbs = row.get("Weight (lbs)")
    if pd.notna(weight_lbs):
        try:
            weight_val = float(weight_lbs)
            if weight_val < 50 or weight_val > 500:
                warnings.append(f"Row {row_num}: Unusual weight {weight_val} lbs (50-500 expected)")
        except (ValueError, TypeError):
            pass
    
    # Validate Height
    height_in = row.get("Height (in)")
    if pd.notna(height_in):
        try:
            height_val = float(height_in)
            if height_val < 48 or height_val > 96:
                warnings.append(f"Row {row_num}: Unusual height {height_val} inches (48-96 expected)")
        except (ValueError, TypeError):
            pass
    
    # Validate TimeSinceSx
    time_since = row.get("TimeSinceSx")
    if pd.notna(time_since):
        try:
            time_val = float(time_since)
            if time_val < 0 or time_val > 120:
                warnings.append(f"Row {row_num}: Unusual time since surgery {time_val} months")
        except (ValueError, TypeError):
            pass
    
    return valid, warnings

# =========================
# ETL CONFIGURATION
# =========================
# Update this path to where your Excel/CSV files are located
DATA_FOLDER = r"C:\path\to\your\data\folder"  # UPDATE THIS!

# Columns that are metadata (not measurements)
META_COLUMNS = [
    "SubjectID", "Date", "TimeSinceSx",
    "Age", "Gender", "Weight (lbs)", "Weight (kg)", 
    "Height (in)", "Height (cm)", "KneeInjury", 
    "DomLimb", "SurgeryLimb"
]

# =========================
# CREATE AUDIT TABLES
# =========================
print("\nSetting up audit trail...")
with engine.connect() as conn:
    # Create data_load_log table
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS data_load_log (
            log_id SERIAL PRIMARY KEY,
            file_name VARCHAR(255) NOT NULL,
            loaded_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
            rows_processed INTEGER,
            rows_succeeded INTEGER,
            rows_failed INTEGER,
            warnings TEXT,
            status VARCHAR(20)
        )
    """))
    
    # Add audit columns to subject table if they don't exist
    result = conn.execute(text("""
        SELECT column_name 
        FROM information_schema.columns 
        WHERE table_name = 'subject'
    """))
    existing_columns = {row[0] for row in result}
    
    if 'created_at' not in existing_columns:
        conn.execute(text("ALTER TABLE subject ADD COLUMN created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP"))
    if 'updated_at' not in existing_columns:
        conn.execute(text("ALTER TABLE subject ADD COLUMN updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP"))
    
    # Add audit columns to test_session table
    result = conn.execute(text("""
        SELECT column_name 
        FROM information_schema.columns 
        WHERE table_name = 'test_session'
    """))
    existing_columns = {row[0] for row in result}
    
    if 'created_at' not in existing_columns:
        conn.execute(text("ALTER TABLE test_session ADD COLUMN created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP"))
    if 'updated_at' not in existing_columns:
        conn.execute(text("ALTER TABLE test_session ADD COLUMN updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP"))
    
    conn.commit()
    print("✅ Audit trail tables ready")

# =========================
# UPDATE SUBJECT TABLE SCHEMA
# =========================
print("\nChecking if subject table needs demographic columns...")
with engine.connect() as conn:
    # Get existing columns
    result = conn.execute(text("""
        SELECT column_name 
        FROM information_schema.columns 
        WHERE table_name = 'subject'
    """))
    existing_columns = {row[0] for row in result}
    
    # Add missing demographic columns
    columns_to_add = []
    if 'age' not in existing_columns:
        columns_to_add.append("ADD COLUMN age INTEGER")
    if 'gender' not in existing_columns:
        columns_to_add.append("ADD COLUMN gender INTEGER")
    if 'weight_lbs' not in existing_columns:
        columns_to_add.append("ADD COLUMN weight_lbs NUMERIC(10,2)")
    if 'weight_kg' not in existing_columns:
        columns_to_add.append("ADD COLUMN weight_kg NUMERIC(10,2)")
    if 'height_in' not in existing_columns:
        columns_to_add.append("ADD COLUMN height_in NUMERIC(10,2)")
    if 'height_cm' not in existing_columns:
        columns_to_add.append("ADD COLUMN height_cm NUMERIC(10,2)")
    if 'knee_injury' not in existing_columns:
        columns_to_add.append("ADD COLUMN knee_injury VARCHAR(50)")
    if 'dom_limb' not in existing_columns:
        columns_to_add.append("ADD COLUMN dom_limb VARCHAR(10)")
    if 'surgery_limb' not in existing_columns:
        columns_to_add.append("ADD COLUMN surgery_limb VARCHAR(10)")
    
    if columns_to_add:
        alter_statement = "ALTER TABLE subject " + ", ".join(columns_to_add)
        conn.execute(text(alter_statement))
        conn.commit()
        print(f"✅ Added {len(columns_to_add)} demographic columns to subject table")
    else:
        print("✅ Subject table already has all demographic columns")

# =========================
# MAIN ETL PROCESS
# =========================
print("\n========== STARTING ETL PROCESS ==========\n")

# Get all Excel and CSV files
files = [f for f in os.listdir(DATA_FOLDER) 
         if f.endswith(('.xlsx', '.xls', '.csv'))]

if not files:
    print(f"⚠️  No files found in {DATA_FOLDER}")
    print("Please update the DATA_FOLDER path!")
else:
    print(f"Found {len(files)} file(s) to process\n")

success = []
failed = []

for file in files:
    rows_processed = 0
    rows_succeeded = 0
    rows_failed = 0
    file_warnings = []
    
    try:
        file_path = os.path.join(DATA_FOLDER, file)
        print(f"Processing: {file}")
        
        # Load file
        if file.endswith('.csv'):
            df = pd.read_csv(file_path)
        else:
            df = pd.read_excel(file_path)
        
        with engine.connect() as conn:
            # Get existing measurements
            existing_measurements = {}
            rows = conn.execute(text("SELECT measurement_id, name FROM measurement"))
            for row in rows:
                existing_measurements[row[1]] = row[0]
            
            # Process each row
            for idx, row in df.iterrows():
                rows_processed += 1
                row_warnings = []
                
                try:
                    subject_id_raw = row.get("SubjectID")
                    if pd.isna(subject_id_raw):
                        rows_failed += 1
                        file_warnings.append(f"Row {idx+2}: Missing SubjectID")
                        continue
                    
                    subject_id = str(subject_id_raw).strip()
                    
                    # Validate demographics
                    valid, row_warnings = validate_demographics(row, idx+2, row_warnings)
                    if row_warnings:
                        file_warnings.extend(row_warnings)
                    
                    # Insert or update subject with demographics
                    result = conn.execute(
                        text("SELECT subject_id FROM subject WHERE subject_id = :sid"),
                        {"sid": subject_id}
                    ).fetchone()
                    
                    demographics = {
                        "sid": subject_id,
                        "age": int(row["Age"]) if pd.notna(row.get("Age")) else None,
                        "gender": int(row["Gender"]) if pd.notna(row.get("Gender")) else None,
                        "weight_lbs": clean_measurement(row.get("Weight (lbs)")),
                        "weight_kg": clean_measurement(row.get("Weight (kg)")),
                        "height_in": clean_measurement(row.get("Height (in)")),
                        "height_cm": clean_measurement(row.get("Height (cm)")),
                        "knee_injury": str(row["KneeInjury"]) if pd.notna(row.get("KneeInjury")) else None,
                        "dom_limb": str(row["DomLimb"]) if pd.notna(row.get("DomLimb")) else None,
                        "surgery_limb": str(row["SurgeryLimb"]) if pd.notna(row.get("SurgeryLimb")) else None
                    }
                    
                    if not result:
                        # Insert new subject
                        conn.execute(text("""
                            INSERT INTO subject 
                            (subject_id, age, gender, weight_lbs, weight_kg, 
                             height_in, height_cm, knee_injury, dom_limb, surgery_limb,
                             created_at, updated_at)
                            VALUES (:sid, :age, :gender, :weight_lbs, :weight_kg,
                                    :height_in, :height_cm, :knee_injury, :dom_limb, :surgery_limb,
                                    CURRENT_TIMESTAMP, CURRENT_TIMESTAMP)
                        """), demographics)
                    else:
                        # Update existing subject with latest demographics
                        conn.execute(text("""
                            UPDATE subject 
                            SET age = :age, gender = :gender, 
                                weight_lbs = :weight_lbs, weight_kg = :weight_kg,
                                height_in = :height_in, height_cm = :height_cm,
                                knee_injury = :knee_injury, dom_limb = :dom_limb, 
                                surgery_limb = :surgery_limb,
                                updated_at = CURRENT_TIMESTAMP
                            WHERE subject_id = :sid
                        """), demographics)
                    
                    # Parse date
                    session_date = parse_date(row.get("Date"))
                    if not session_date:
                        rows_failed += 1
                        file_warnings.append(f"Row {idx+2}: Invalid date")
                        continue
                    
                    # Check if session already exists
                    existing_session = conn.execute(text("""
                        SELECT session_id FROM test_session 
                        WHERE subject_id = :subject_id AND session_date = :session_date
                    """), {
                        "subject_id": subject_id,
                        "session_date": session_date
                    }).fetchone()
                    
                    if existing_session:
                        # Update existing session
                        session_id = existing_session[0]
                        conn.execute(text("""
                            UPDATE test_session 
                            SET time_since_sx = :time_since_sx,
                                updated_at = CURRENT_TIMESTAMP
                            WHERE session_id = :session_id
                        """), {
                            "session_id": session_id,
                            "time_since_sx": clean_measurement(row.get("TimeSinceSx"))
                        })
                        
                        # Delete old measurements for this session (will re-insert)
                        conn.execute(text("""
                            DELETE FROM session_measurement WHERE session_id = :session_id
                        """), {"session_id": session_id})
                    else:
                        # Insert new session
                        session_id = conn.execute(text("""
                            INSERT INTO test_session (subject_id, session_date, time_since_sx,
                                                      created_at, updated_at)
                            VALUES (:subject_id, :session_date, :time_since_sx,
                                    CURRENT_TIMESTAMP, CURRENT_TIMESTAMP)
                            RETURNING session_id
                        """), {
                            "subject_id": subject_id,
                            "session_date": session_date,
                            "time_since_sx": clean_measurement(row.get("TimeSinceSx"))
                        }).scalar()
                    
                    # Insert measurements (skip metadata columns)
                    measurements_added = 0
                    for col in df.columns:
                        if col in META_COLUMNS:
                            continue
                        
                        value = clean_measurement(row[col])
                        if value is None:
                            continue
                        
                        # Validate measurement value
                        if value < -10000 or value > 10000:
                            file_warnings.append(f"Row {idx+2}, Column {col}: Unusual value {value}")
                        
                        # Create measurement type if doesn't exist
                        if col not in existing_measurements:
                            m_id = conn.execute(text("""
                                INSERT INTO measurement (name)
                                VALUES (:name)
                                RETURNING measurement_id
                            """), {"name": col}).scalar()
                            existing_measurements[col] = m_id
                        else:
                            m_id = existing_measurements[col]
                        
                        # Insert session measurement
                        conn.execute(text("""
                            INSERT INTO session_measurement
                            (session_id, measurement_id, value)
                            VALUES (:session_id, :measurement_id, :value)
                        """), {
                            "session_id": session_id,
                            "measurement_id": m_id,
                            "value": value
                        })
                        measurements_added += 1
                    
                    rows_succeeded += 1
                    conn.commit()
                    
                except Exception as row_error:
                    rows_failed += 1
                    file_warnings.append(f"Row {idx+2}: {str(row_error)}")
                    conn.rollback()
            
            # Log the data load
            conn.execute(text("""
                INSERT INTO data_load_log 
                (file_name, rows_processed, rows_succeeded, rows_failed, warnings, status)
                VALUES (:file_name, :rows_processed, :rows_succeeded, :rows_failed, :warnings, :status)
            """), {
                "file_name": file,
                "rows_processed": rows_processed,
                "rows_succeeded": rows_succeeded,
                "rows_failed": rows_failed,
                "warnings": "\n".join(file_warnings) if file_warnings else None,
                "status": "SUCCESS" if rows_failed == 0 else "PARTIAL"
            })
            conn.commit()
        
        print(f"✅ Successfully processed {file}")
        print(f"   Rows: {rows_succeeded}/{rows_processed} succeeded")
        if file_warnings:
            print(f"   ⚠️  {len(file_warnings)} warning(s) - check data_load_log table")
        success.append(file)
        
    except Exception as e:
        print(f"❌ Failed processing {file}: {e}")
        failed.append((file, str(e)))
        
        # Log the failed load
        with engine.connect() as conn:
            conn.execute(text("""
                INSERT INTO data_load_log 
                (file_name, rows_processed, rows_succeeded, rows_failed, warnings, status)
                VALUES (:file_name, :rows_processed, :rows_succeeded, :rows_failed, :warnings, :status)
            """), {
                "file_name": file,
                "rows_processed": rows_processed,
                "rows_succeeded": rows_succeeded,
                "rows_failed": rows_failed,
                "warnings": str(e),
                "status": "FAILED"
            })
            conn.commit()

# =========================
# SUMMARY
# =========================
print("\n========== ETL SUMMARY ==========")

print(f"\n✅ Loaded ({len(success)}):")
for f in success:
    print("  ", f)

print(f"\n❌ Failed ({len(failed)}):")
for f, err in failed:
    print("  ", f, "→", err)

print("\n📊 View detailed logs:")
print("   SELECT * FROM data_load_log ORDER BY loaded_at DESC;")

print("\n🎉 ETL Complete.")